# Sequential Quality-Aware Refinement LDM for CT→MRI

### Two-Stage Architecture:
```
Stage 1 (LDM):
  Sequential CT slices → Draft MRI slices
  
Stage 2 (Quality-Aware Refiner):
  1. 양옆 ±5 슬라이스의 draft 중 좋은 것 5개 선택 (Learnable)
  2. Selected drafts → Context features
  3. Center draft + CT + Context → Refined MRI (Attention U-Net)
```

In [3]:
import sys
import os
from pathlib import Path
import warnings
import json
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# LDM path
ldm_path = os.path.expanduser('~/CT2MRI/diffusion_model/ldm')
if ldm_path not in sys.path:
    sys.path.insert(0, ldm_path)

# 원본 LDM imports
from ldm.modules.diffusionmodules.openaimodel import UNetModel
from ldm.modules.diffusionmodules.util import (
    make_beta_schedule,
    make_ddim_timesteps,
    make_ddim_sampling_parameters
)

# 원본 LDM 함수들을 import (latent_diffusion.ipynb에서)
# 실제로는 latent_diffusion.ipynb를 .py로 변환하거나 
# 필요한 함수들을 직접 복사해야 함

warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")


PyTorch: 2.5.1+cu124
CUDA: True


In [13]:

# =============================================================================
# Config
# =============================================================================

CFG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
    "data_root": "../data/processed_with_skullstrip",
    "img_size": 256,
    "train_ratio": 0.9,
    "context_range": 5,
    "num_selected": 5,
    
    # ⭐ 원본 LDM checkpoint (VAE, UNet, cond_enc 모두 포함)
    "ldm_checkpoint": "./ckpt_ct2mri_b0_ldm_strip/last.pt",
    
    # ⭐ VAE config & checkpoint (원본 LDM에서 사용)
    "vae_config": "autoencoder_kl_64x64x3.yaml",
    "vae_ckpt": "ae_model.ckpt",
    
    # Training (Stage 2만)
    "batch_size": 16,
    "num_workers": 4,
    "epochs": 100,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    
    # Sampling (원본 LDM 설정과 동일)
    "num_timesteps": 1000,
    "beta_schedule": "cosine",
    "ddim_steps": 200,
    "ddim_eta": 0.0,
    
    # Checkpointing
    "save_dir": "./ckpt_sequential_refinement",
    "save_every": 5,
}

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
Path(CFG["save_dir"]).mkdir(parents=True, exist_ok=True)

print("\nConfiguration:")
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

with open(Path(CFG["save_dir"]) / "config.json", 'w') as f:
    json.dump(CFG, f, indent=2)



Configuration:
  device              : cuda
  seed                : 42
  data_root           : ../data/processed_with_skullstrip
  img_size            : 256
  train_ratio         : 0.9
  context_range       : 5
  num_selected        : 5
  ldm_checkpoint      : ./ckpt_ct2mri_b0_ldm_strip/last.pt
  vae_config          : autoencoder_kl_64x64x3.yaml
  vae_ckpt            : ae_model.ckpt
  batch_size          : 16
  num_workers         : 4
  epochs              : 100
  lr                  : 0.0001
  weight_decay        : 0.0001
  grad_clip           : 1.0
  num_timesteps       : 1000
  beta_schedule       : cosine
  ddim_steps          : 200
  ddim_eta            : 0.0
  save_dir            : ./ckpt_sequential_refinement
  save_every          : 5


In [14]:
# =============================================================================
# 원본 LDM 함수들 (latent_diffusion.ipynb에서 복사)
# =============================================================================

from ldm.util import instantiate_from_config
from ldm.models.autoencoder import AutoencoderKL, VQModelInterface
from omegaconf import OmegaConf

class VAEWrapper(nn.Module):
    """원본 LDM의 VAE wrapper"""
    def __init__(self, encode_fn, decode_fn):
        super().__init__()
        self._enc = encode_fn
        self._dec = decode_fn

    @torch.no_grad()
    def encode(self, x):
        return self._enc(x)

    @torch.no_grad()
    def decode(self, z):
        return self._dec(z)


def build_vae(vae_config_path, vae_ckpt_path, device):
    """원본 LDM의 build_vae 함수"""
    raw = OmegaConf.load(vae_config_path)
    conf = OmegaConf.to_container(raw, resolve=True)
    
    assert os.path.exists(vae_ckpt_path), f"VAE ckpt not found: {vae_ckpt_path}"
    
    top = conf.get("model", {})
    top_target = top.get("target", "")
    
    if "AutoencoderKL" in top_target:
        kl = instantiate_from_config(top)
        kl.to(device)
        sd_all = torch.load(vae_ckpt_path, map_location="cpu")
        state = sd_all.get("state_dict", sd_all)
        kl.load_state_dict(state, strict=False)
        kl.eval()
        
        for p in kl.parameters():
            p.requires_grad_(False)
        
        scale = 0.18215
        
        def _enc(x_m11):
            x = x_m11.repeat(1, 3, 1, 1) if x_m11.size(1) == 1 else x_m11
            posterior = kl.encode(x)
            z = posterior.sample() * scale
            return z
        
        def _dec(z):
            x = kl.decode(z / scale)
            return x.mean(1, keepdim=True) if x.size(1) == 3 else x
        
        return VAEWrapper(_enc, _dec)
    else:
        raise NotImplementedError("Only AutoencoderKL supported")


class CTCondEncoder(nn.Module):
    """원본 LDM의 CT condition encoder"""
    def __init__(self, cond_channels=1):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, 2, 1), nn.SiLU(),      # H/2
            nn.Conv2d(32, 64, 3, 2, 1), nn.SiLU(),     # H/4
            nn.Conv2d(64, cond_channels, 1)            # channel match
        )
    
    def forward(self, x):
        return self.enc(x)


@torch.no_grad()
def sample_ddim_latent(model, cond_enc, vae_wrap, ct_m11, sched_dict, ddim_steps=200, eta=0.0, device="cpu"):
    """원본 LDM의 DDIM sampling 함수"""
    B = ct_m11.size(0)
    ct_cond = cond_enc(ct_m11)
    
    # Latent shape
    with torch.no_grad():
        z_test = vae_wrap.encode(ct_m11)
        Cz = z_test.shape[1]
        lat_h, lat_w = z_test.shape[-2], z_test.shape[-1]
    
    z = torch.randn(B, Cz, lat_h, lat_w, device=device)
    
    ts = np.array(make_ddim_timesteps('uniform', ddim_steps, len(sched_dict['betas']))) - 1
    sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(
        sched_dict['alphas_cumprod'].cpu().numpy(), 
        ts, 
        eta
    )
    
    sigmas = torch.from_numpy(sigmas).float().to(device)
    alphas = torch.from_numpy(alphas).float().to(device)
    alphas_prev = torch.from_numpy(alphas_prev).float().to(device)
    
    for i in reversed(range(len(ts))):
        t = int(ts[i])
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = model(torch.cat([z, ct_cond], 1).float(), t_b)
        a_t, a_prev, s_t = alphas[i], alphas_prev[i], sigmas[i]
        z0 = (z - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
        dir_term = torch.sqrt(1.0 - a_prev - s_t**2) * eps
        noise = s_t * torch.randn_like(z) if i > 0 else 0.0
        z = torch.sqrt(a_prev) * z0 + dir_term + noise
    
    x_pred_m11 = vae_wrap.decode(z)
    return x_pred_m11



In [15]:

# =============================================================================
# Dataset
# =============================================================================

class SequentialDataset(Dataset):
    def __init__(self, data_root, context_range=5, img_size=256, mode='train', train_ratio=0.9):
        self.data_root = Path(data_root)
        self.context_range = context_range
        self.img_size = img_size
        self.mode = mode
        
        print(f"\n[{mode.upper()}] Loading dataset...")
        self.patient_slices = self._collect()
        print(f"  Found {len(self.patient_slices)} patients")
        
        patients = sorted(self.patient_slices.keys())
        n_train = int(len(patients) * train_ratio)
        self.patients = patients[:n_train] if mode == 'train' else patients[n_train:]
        
        self.samples = []
        for patient in self.patients:
            slices = self.patient_slices[patient]
            n = len(slices['ct'])
            for i in range(context_range, n - context_range):
                self.samples.append((patient, i))
        
        print(f"  Total samples: {len(self.samples)}")
        print(f"  Patients: {len(self.patients)}\n")
    
    def _collect(self):
        data = {}
        for group in ['bleed', 'non_bleed']:
            group_dir = self.data_root / group
            if not group_dir.exists():
                continue
            for patient_dir in sorted(group_dir.iterdir()):
                if not patient_dir.is_dir():
                    continue
                patient_name = f"{group}_{patient_dir.name}"
                ct_dir = patient_dir / 'ct_axial'
                mri_dir = patient_dir / 'mri_b0_axial'
                if not (ct_dir.exists() and mri_dir.exists()):
                    continue
                ct_files = sorted(ct_dir.glob('*.png'))
                mri_files = sorted(mri_dir.glob('*.png'))
                ct_dict = {f.stem: f for f in ct_files}
                mri_dict = {f.stem: f for f in mri_files}
                common = sorted(set(ct_dict.keys()) & set(mri_dict.keys()))
                if len(common) < 2 * self.context_range + 3:
                    continue
                data[patient_name] = {
                    'ct': [ct_dict[s] for s in common],
                    'mri': [mri_dict[s] for s in common]
                }
        return data
    
    def _load(self, path):
        img = Image.open(path).convert('L').resize(
            (self.img_size, self.img_size), Image.BICUBIC
        )
        return np.array(img, dtype=np.float32) / 255.0
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        patient, center_idx = self.samples[idx]
        slices = self.patient_slices[patient]
        
        mri_center = self._load(slices['mri'][center_idx])
        
        ct_seq = []
        for offset in range(-self.context_range, self.context_range + 1):
            ct = self._load(slices['ct'][center_idx + offset])
            ct_seq.append(ct)
        
        ct_seq_t = torch.stack(
            [torch.from_numpy(c).unsqueeze(0) * 2 - 1 for c in ct_seq], 
            dim=0
        )
        mri_t = torch.from_numpy(mri_center).unsqueeze(0) * 2 - 1
        
        return {
            'ct_sequence': ct_seq_t,
            'mri_center': mri_t,
            'patient': patient,
            'slice_idx': center_idx
        }


In [16]:
# =============================================================================
# Stage 1: 원본 LDM Wrapper (Frozen)
# =============================================================================

class OriginalLDM(nn.Module):
    """원본 latent_diffusion.ipynb를 그대로 사용"""
    
    def __init__(self, ldm_ckpt_path, vae_config_path, vae_ckpt_path, device):
        super().__init__()
        self.device = device
        
        print("\n" + "="*70)
        print("Loading Original LDM (from latent_diffusion.ipynb)")
        print("="*70)
        
        # 1. VAE
        print(f"\n[1/4] Loading VAE...")
        print(f"  Config: {vae_config_path}")
        print(f"  Ckpt: {vae_ckpt_path}")
        self.vae_wrap = build_vae(vae_config_path, vae_ckpt_path, device)
        print("  ✓ VAE loaded")
        
        # 2. LDM checkpoint
        print(f"\n[2/4] Loading LDM checkpoint: {ldm_ckpt_path}")
        ckpt = torch.load(ldm_ckpt_path, map_location='cpu')
        
        cfg_orig = ckpt.get('cfg', {})
        lu_cfg = cfg_orig.get('latent_unet', {})
        
        z_ch = lu_cfg.get('z_channels', 4)
        cond_ch = lu_cfg.get('cond_channels', 1)
        
        print(f"  z_channels: {z_ch}")
        print(f"  cond_channels: {cond_ch}")
        
        # 3. UNet
        print(f"\n[3/4] Building UNet...")
        self.unet = UNetModel(
            image_size=32,
            in_channels=z_ch + cond_ch,
            out_channels=z_ch,
            model_channels=lu_cfg.get('model_channels', 192),
            num_res_blocks=lu_cfg.get('num_res_blocks', 2),
            channel_mult=tuple(lu_cfg.get('channel_mult', (1,2,3,4))),
            attention_resolutions=lu_cfg.get('attention_resolutions', (16,)),
            num_heads=lu_cfg.get('num_heads', -1),
            num_head_channels=lu_cfg.get('num_head_channels', 64),
            dropout=lu_cfg.get('dropout', 0.1),
            use_scale_shift_norm=lu_cfg.get('use_scale_shift_norm', True),
            use_checkpoint=False,
        )
        
        self.unet.load_state_dict(ckpt['model'], strict=True)
        self.unet.to(device).eval()
        for p in self.unet.parameters():
            p.requires_grad = False
        print("  ✓ UNet loaded")
        
        # 4. CT Condition Encoder
        print(f"\n[4/4] Building CT Condition Encoder...")
        self.cond_enc = CTCondEncoder(cond_channels=cond_ch)
        self.cond_enc.load_state_dict(ckpt['cond_enc'], strict=True)
        self.cond_enc.to(device).eval()
        for p in self.cond_enc.parameters():
            p.requires_grad = False
        print("  ✓ CT Cond Encoder loaded")
        
        # Diffusion schedule
        num_timesteps = cfg_orig.get('num_timesteps', 1000)
        beta_schedule = cfg_orig.get('beta_schedule', 'cosine')
        
        betas = torch.from_numpy(
            make_beta_schedule(beta_schedule, num_timesteps)
        ).float().to(device)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        
        self.sched_dict = {
            'betas': betas,
            'alphas': alphas,
            'alphas_cumprod': alphas_cumprod
        }
        
        print(f"\n  ✓ Schedule (T={num_timesteps}, {beta_schedule})")
        print("\n" + "="*70)
        print("✓ Original LDM Ready!")
        print("="*70 + "\n")
    
    @torch.no_grad()
    def generate_draft(self, ct_m11, ddim_steps=200, eta=0.0):
        """원본 LDM으로 draft 생성"""
        return sample_ddim_latent(
            self.unet, 
            self.cond_enc, 
            self.vae_wrap, 
            ct_m11,
            self.sched_dict,
            ddim_steps=ddim_steps,
            eta=eta,
            device=self.device
        )

In [17]:
# =============================================================================
# Stage 2: Quality Selector + Refiner (Trainable)
# =============================================================================

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q = nn.Conv2d(channels, channels // 8, 1)
        self.k = nn.Conv2d(channels, channels // 8, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        B, C, H, W = x.shape
        q = self.q(x).view(B, -1, H * W).permute(0, 2, 1)
        k = self.k(x).view(B, -1, H * W)
        v = self.v(x).view(B, -1, H * W)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(B, C, H, W)
        return x + self.gamma * out


class CrossAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q = nn.Conv2d(channels, channels // 8, 1)
        self.k = nn.Conv2d(channels, channels // 8, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
    
    def forward(self, query, key_value):
        B, C, H, W = query.shape
        q = self.q(query).view(B, -1, H * W).permute(0, 2, 1)
        k = self.k(key_value).view(B, -1, H * W)
        v = self.v(key_value).view(B, -1, H * W)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(B, C, H, W)
        return query + self.gamma * out


class SelfSupervisedQualitySelector(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.AdaptiveAvgPool2d(1),
        )
        self.scorer = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    
    def _sobel(self, x):
        sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
        sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
        gx = F.conv2d(x, sobel_x, padding=1)
        gy = F.conv2d(x, sobel_y, padding=1)
        return torch.sqrt(gx**2 + gy**2 + 1e-8)
    
    def _consistency(self, draft, neighbors):
        scores = []
        draft_grad = self._sobel(draft)
        for nb in neighbors:
            nb_grad = self._sobel(nb)
            sim = F.cosine_similarity(
                draft_grad.flatten(1), 
                nb_grad.flatten(1), 
                dim=1
            )
            scores.append(sim)
        return torch.stack(scores, dim=1).mean(dim=1)
    
    def forward(self, draft_sequence):
        B, N = draft_sequence.shape[:2]
        quality_scores = []
        
        for n in range(N):
            draft = draft_sequence[:, n]
            nb_idx = [i for i in range(max(0,n-2), min(N,n+3)) if i != n]
            neighbors = [draft_sequence[:, i] for i in nb_idx]
            
            consistency = self._consistency(draft, neighbors)
            neural_q = self.scorer(self.encoder(draft)).squeeze(1)
            quality = 0.6 * neural_q + 0.4 * consistency
            quality_scores.append(quality)
        
        return torch.stack(quality_scores, dim=1)


class AttentionUNet(nn.Module):
    def __init__(self, in_channels, out_channels=1):
        super().__init__()
        self.enc1 = self._block(in_channels, 64)
        self.enc2 = self._block(64, 128)
        self.enc3 = self._block(128, 256)
        self.enc4 = self._block(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(
            self._block(512, 1024),
            SelfAttention(1024)
        )
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = self._block(1024, 512)
        self.attn4 = CrossAttention(512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = self._block(512, 256)
        self.attn3 = CrossAttention(256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = self._block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = self._block(128, 64)
        self.out = nn.Conv2d(64, out_channels, 1)
    
    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True),
        )
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], 1)
        d4 = self.dec4(d4)
        d4 = self.attn4(d4, e4)
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], 1)
        d3 = self.dec3(d3)
        d3 = self.attn3(d3, e3)
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], 1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], 1)
        d1 = self.dec1(d1)
        return self.out(d1)

In [18]:
# =============================================================================
# Full Model
# =============================================================================

class SequentialRefinementLDM(nn.Module):
    def __init__(self, original_ldm, num_selected=5, ddim_steps=200):
        super().__init__()
        
        self.original_ldm = original_ldm
        self.num_selected = num_selected
        self.ddim_steps = ddim_steps
        
        print("Building Stage 2 (Trainable)...")
        
        self.quality_selector = SelfSupervisedQualitySelector()
        print("  ✓ Quality Selector")
        
        self.context_encoder = nn.Sequential(
            nn.Conv2d(num_selected, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
        )
        print("  ✓ Context Encoder")
        
        self.refiner = AttentionUNet(in_channels=258, out_channels=1)
        print("  ✓ Refiner")
        
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"\nParams: Trainable={trainable:,} ({trainable/total*100:.1f}%), Total={total:,}\n")
    
    @torch.no_grad()
    def generate_drafts(self, ct_sequence):
        """Stage 1: 원본 LDM으로 draft 생성"""
        B, N = ct_sequence.shape[:2]
        drafts = []
        
        for n in range(N):
            ct_slice = ct_sequence[:, n]
            draft = self.original_ldm.generate_draft(ct_slice, self.ddim_steps, eta=0.0)
            drafts.append(draft)
        
        return torch.stack(drafts, dim=1)
    
    def forward(self, ct_sequence, return_intermediates=False):
        B, N = ct_sequence.shape[:2]
        
        # Stage 1: Draft
        draft_sequence = self.generate_drafts(ct_sequence)
        
        center_idx = N // 2
        draft_center = draft_sequence[:, center_idx]
        ct_center = ct_sequence[:, center_idx]
        
        # Stage 2-1: Quality selection
        quality_scores = self.quality_selector(draft_sequence)
        _, top_indices = torch.topk(quality_scores, k=self.num_selected, dim=1)
        
        selected_drafts = []
        for b in range(B):
            selected = draft_sequence[b, top_indices[b]]
            selected_drafts.append(selected)
        selected_drafts = torch.stack(selected_drafts, dim=0)
        
        # Stage 2-2: Context encoding
        context_features = self.context_encoder(selected_drafts.squeeze(2))
        
        # Stage 2-3: Refinement
        refiner_input = torch.cat([draft_center, ct_center, context_features], dim=1)
        refined_mri = self.refiner(refiner_input)
        
        if return_intermediates:
            return refined_mri, draft_center, top_indices
        return refined_mri, draft_center



In [19]:
# =============================================================================
# Loss & Training
# =============================================================================

class RefinementLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True)
        self.vgg = vgg.features[:16].eval()
        for p in self.vgg.parameters():
            p.requires_grad = False
        print("✓ RefinementLoss initialized")
    
    def forward(self, refined, draft, target, epoch=0):
        l1 = F.l1_loss(refined, target)
        l2 = F.mse_loss(refined, target)
        recon = 0.7 * l1 + 0.3 * l2
        
        perc = torch.tensor(0.0, device=refined.device)
        if epoch >= 20:
            r3 = refined.repeat(1,3,1,1)
            t3 = target.repeat(1,3,1,1)
            mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(refined.device)
            std = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(refined.device)
            with torch.no_grad():
                tf = self.vgg((t3 - mean) / std)
            rf = self.vgg((r3 - mean) / std)
            perc = F.mse_loss(rf, tf)
        
        improve = torch.tensor(0.0, device=refined.device)
        if epoch >= 20:
            de = F.l1_loss(draft, target, reduction='none').mean(dim=(1,2,3))
            re = F.l1_loss(refined, target, reduction='none').mean(dim=(1,2,3))
            improve = F.relu(re - de + 0.01).mean()
        
        if epoch < 20:
            w = {'recon': 1.0, 'perc': 0.0, 'improve': 0.0}
        elif epoch < 50:
            w = {'recon': 1.0, 'perc': 0.3, 'improve': 0.1}
        else:
            w = {'recon': 1.0, 'perc': 0.5, 'improve': 0.2}
        
        total = w['recon']*recon + w['perc']*perc + w['improve']*improve
        
        return {
            'total': total, 'recon': recon, 'l1': l1, 'l2': l2,
            'perceptual': perc, 'improvement': improve
        }


def train_epoch(model, loader, criterion, optimizer, epoch, device, grad_clip):
    model.train()
    model.original_ldm.eval()  # Keep Stage 1 frozen
    
    losses = []
    metrics = defaultdict(list)
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [Train]")
    
    for batch in pbar:
        ct_seq = batch['ct_sequence'].to(device)
        mri_target = batch['mri_center'].to(device)
        
        optimizer.zero_grad()
        
        refined, draft = model(ct_seq)
        
        loss_dict = criterion(refined, draft, mri_target, epoch)
        
        loss = loss_dict['total']
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        optimizer.step()
        
        losses.append(loss.item())
        for k, v in loss_dict.items():
            metrics[k].append(v.item())
        
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'recon': f"{loss_dict['recon'].item():.4f}"
        })
    
    return {k: np.mean(v) for k, v in metrics.items()}


@torch.no_grad()
def val_epoch(model, loader, criterion, epoch, device):
    model.eval()
    
    losses = []
    metrics = defaultdict(list)
    
    for batch in tqdm(loader, desc=f"Epoch {epoch+1} [Val]"):
        ct_seq = batch['ct_sequence'].to(device)
        mri_target = batch['mri_center'].to(device)
        
        refined, draft = model(ct_seq)
        
        loss_dict = criterion(refined, draft, mri_target, epoch)
        
        losses.append(loss_dict['total'].item())
        for k, v in loss_dict.items():
            metrics[k].append(v.item())
    
    return {k: np.mean(v) for k, v in metrics.items()}

In [ ]:
# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":
    # Build Original LDM (Stage 1)
    original_ldm = OriginalLDM(
        CFG["ldm_checkpoint"],
        CFG["vae_config"],
        CFG["vae_ckpt"],
        CFG["device"]
    ).to(CFG["device"])
    
    # Build Full Model
    model = SequentialRefinementLDM(
        original_ldm,
        CFG["num_selected"],
        CFG["ddim_steps"]
    ).to(CFG["device"])
    
    criterion = RefinementLoss().to(CFG["device"])
    
    # Prepare data
    train_ds = SequentialDataset(
        CFG["data_root"],
        CFG["context_range"],
        CFG["img_size"],
        'train',
        CFG["train_ratio"]
    )
    
    val_ds = SequentialDataset(
        CFG["data_root"],
        CFG["context_range"],
        CFG["img_size"],
        'val',
        CFG["train_ratio"]
    )
    
    train_loader = DataLoader(
        train_ds,
        batch_size=CFG["batch_size"],
        shuffle=True,
        num_workers=CFG["num_workers"],
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_ds,
        batch_size=CFG["batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"]
    )
    
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"]
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG["epochs"],
        eta_min=1e-6
    )
    
    # Training loop
    best_val_loss = float('inf')
    history = defaultdict(list)
    
    print("\n" + "="*70)
    print("Starting Training")
    print("="*70 + "\n")
    
    for epoch in range(CFG["epochs"]):
        
        # Train
        train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, 
            epoch, CFG["device"], CFG["grad_clip"]
        )
        
        # Validation
        val_metrics = val_epoch(model, val_loader, criterion, epoch, CFG["device"])
        
        scheduler.step()
        
        # Record history
        for k in train_metrics.keys():
            history[f'train_{k}'].append(train_metrics[k])
            history[f'val_{k}'].append(val_metrics[k])
        
        # Print summary
        print(f"\nEpoch {epoch+1}/{CFG['epochs']}")
        print(f"  Train Loss: {train_metrics['total']:.4f}")
        print(f"  Val Loss:   {val_metrics['total']:.4f}")
        print(f"    ├─ Recon:  {val_metrics['recon']:.4f}")
        print(f"    ├─ Percep: {val_metrics['perceptual']:.4f}")
        print(f"    └─ Improv: {val_metrics['improvement']:.4f}")
        print(f"  LR: {scheduler.get_last_lr()[0]:.6f}")
        
        # Save best
        if val_metrics['total'] < best_val_loss:
            best_val_loss = val_metrics['total']
            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'config': CFG,
                'val_loss': val_metrics['total'],
                'history': dict(history)
            }, Path(CFG["save_dir"]) / 'best_model.pt')
            print(f"  ✓ Saved best model (val_loss: {val_metrics['total']:.4f})")
        
        # Save checkpoint
        if (epoch + 1) % CFG["save_every"] == 0:
            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'config': CFG,
                'history': dict(history)
            }, Path(CFG["save_dir"]) / f'checkpoint_epoch{epoch+1}.pt')
            print(f"  ✓ Saved checkpoint")
        
        print("="*70)
    
    print("\n✓ Training complete!")


Loading Original LDM (from latent_diffusion.ipynb)

[1/4] Loading VAE...
  Config: autoencoder_kl_64x64x3.yaml
  Ckpt: ae_model.ckpt
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
  ✓ VAE loaded

[2/4] Loading LDM checkpoint: ./ckpt_ct2mri_b0_ldm_strip/last.pt
  z_channels: 3
  cond_channels: 1

[3/4] Building UNet...
  ✓ UNet loaded

[4/4] Building CT Condition Encoder...
  ✓ CT Cond Encoder loaded

  ✓ Schedule (T=1000, cosine)

✓ Original LDM Ready!

Building Stage 2 (Trainable)...
  ✓ Quality Selector
  ✓ Context Encoder
  ✓ Refiner

Params: Trainable=33,387,269 (15.0%), Total=221,909,833

✓ RefinementLoss initialized

[TRAIN] Loading dataset...
  Found 360 patients
  Total samples: 1973
  Patients: 324


[VAL] Loading dataset...
  Found 360 patients
  Total samples: 204
  Patients: 36

Epoch 1 [Train]:   6%|▌         | 7/123 [10:36<2:55:42, 90.88s/it, loss=0.8256, recon=0.8256]